[Reference](https://pub.towardsai.net/i-spent-3-months-building-ra-systems-before-learning-these-11-strategies-1a8f6b4278aa)

In [1]:
# Naive RAG approach
def naive_rag(query: str) -> str:
    # 1. Embed the query
    query_embedding = embed(query)

    # 2. Find similar chunks
    chunks = vector_db.search(query_embedding, top_k=5)

    # 3. Generate answer
    context = "\n".join(chunks)
    answer = llm.generate(f"Context: {context}\n\nQuestion: {query}")

    return answer

# Strategy 1: Context-Aware Chunking

In [3]:
!pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.4/275.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.0/223.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 34.0 MB/s eta 0:00:00
   ━━

In [4]:
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

class SmartChunker:
    def __init__(self, max_tokens=512):
        # Use actual tokenizer, not character counts
        self.tokenizer = AutoTokenizer.from_pretrained(
            "sentence-transformers/all-MiniLM-L6-v2"
        )
        self.chunker = HybridChunker(
            tokenizer=self.tokenizer,
            max_tokens=max_tokens,
            merge_peers=True  # Combine small adjacent chunks
        )

    def chunk_document(self, document):
        # Analyzes document structure (headings, paragraphs, tables)
        chunks = list(self.chunker.chunk(dl_doc=document))

        # Each chunk includes heading context
        contextualized_chunks = []
        for chunk in chunks:
            # Adds hierarchical heading information
            contextualized_text = self.chunker.contextualize(chunk=chunk)
            contextualized_chunks.append(contextualized_text)

        return contextualized_chunks

# Strategy 2: Contextual Retrieval

In [5]:
async def enrich_chunk(chunk: str, document: str, title: str) -> str:
    """Add contextual prefix using LLM"""
    prompt = f"""
Title: {title}
{document[:4000]}
{chunk}

Provide brief context (1-2 sentences) explaining what this chunk discusses
in relation to the full document. Format: "This chunk from [title] discusses [explanation]." """
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=150
    )

    context = response.choices[0].message.content.strip()

    # Embed the contextualized version
    return f"{context}\n\n{chunk}"

# Strategy 3: Re-ranking

In [6]:
from sentence_transformers import CrossEncoder

# Initialize once
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
async def search_with_reranking(query: str, limit: int = 5) -> list:
    # Stage 1: Fast vector retrieval (get 4x candidates)
    candidate_limit = min(limit * 4, 20)
    query_embedding = await embedder.embed_query(query)

    candidates = await db.query(
        "SELECT content, metadata FROM chunks ORDER BY embedding  $1 LIMIT $2",
        query_embedding, candidate_limit
    )

    # Stage 2: Re-rank with cross-encoder
    pairs = [[query, row['content']] for row in candidates]
    scores = reranker.predict(pairs)

    # Sort by reranker scores and return top N
    reranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )[:limit]

    return [doc for doc, score in reranked]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

# Strategy 4: Query Expansion

In [ ]:
async def expand_query(query: str) -> str:
    """Expand brief query into detailed version"""
    system_prompt = """You are a query expansion assistant.
Take brief user queries and expand them into more detailed versions that:
1. Add relevant context and clarifications
2. Include related terminology and concepts
3. Specify what aspects should be covered
4. Maintain the original intent
5. Keep it as a single, coherent question
Expand the query to be 2-3x more detailed while staying focused."""
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Expand this query: {query}"}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content.strip()

# Strategy 5: Multi-Query RAG

In [7]:
async def search_with_multi_query(query: str, limit: int = 5) -> list:
    # Generate query variations
    variations_prompt = f"""Generate 3 different phrasings of this query:
    "{query}"

    Return only the 3 queries, one per line."""

    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": variations_prompt}],
        temperature=0.7
    )

    queries = [query] + response.choices[0].message.content.strip().split('\n')

    # Execute all searches in parallel
    search_tasks = []
    for q in queries:
        query_embedding = await embedder.embed_query(q)
        task = db.fetch(
            "SELECT * FROM match_chunks($1::vector, $2)",
            query_embedding, limit
        )
        search_tasks.append(task)

    results_lists = await asyncio.gather(*search_tasks)

    # Deduplicate by chunk ID, keeping highest similarity
    seen = {}
    for results in results_lists:
        for row in results:
            chunk_id = row['chunk_id']
            if chunk_id not in seen or row['similarity'] > seen[chunk_id]['similarity']:
                seen[chunk_id] = row

    # Return top N unique results
    return sorted(
        seen.values(),
        key=lambda x: x['similarity'],
        reverse=True
    )[:limit]

# Strategy 6: Agentic RAG

In [9]:
!pip install pydantic_ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.4/93.4 kB 4.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/ba

In [1]:
from pydantic_ai import Agent
agent = Agent(
    'openai:gpt-4o',
    system_prompt='You are a RAG assistant with multiple retrieval tools. Choose the right tool(s) for each query.'
)
@agent.tool
async def search_knowledge_base(query: str, limit: int = 5) -> str:
    """Semantic search over document chunks"""
    query_embedding = await embedder.embed_query(query)
    results = await db.match_chunks(query_embedding, limit)
    return format_results(results)
@agent.tool
async def retrieve_full_document(document_title: str) -> str:
    """Retrieve complete document when chunks lack context"""
    result = await db.query(
        "SELECT title, content FROM documents WHERE title ILIKE %s",
        f"%{document_title}%"
    )
    return f"**{result['title']}**\n\n{result['content']}"
@agent.tool
async def sql_query(question: str) -> str:
    """Query structured database for specific data"""
    # Agent can write SQL queries for structured data
    # (In production, use proper SQL generation with safety checks)
    return execute_safe_sql(question)

# Strategy 7: Self-Reflective RAG

In [3]:
async def search_with_self_reflection(query: str, limit: int = 5, max_iterations: int = 2) -> dict:
    """Self-correcting search loop"""

    for iteration in range(max_iterations):
        # Perform search
        results = await vector_search(query, limit)

        # Grade relevance
        grade_prompt = f"""Query: {query}

Retrieved documents:
{format_docs_for_grading(results)}
Grade the relevance of these documents to the query on a scale of 1-5.
Respond with only the number."""
        grade_response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": grade_prompt}],
            temperature=0
        )

        grade = int(grade_response.choices[0].message.content.strip().split()[0])

        # If good results, return them
        if grade >= 3:
            return {
                "results": results,
                "iterations": iteration + 1,
                "final_query": query
            }

        # If poor results and not last iteration, refine query
        if iteration             refine_prompt = f"""Query "{query}" returned low-relevance results.

Suggest an improved query that might find better documents.
Respond with only the improved query."""
            refined_response = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": refine_prompt}],
                temperature=0.5
            )

            query = refined_response.choices[0].message.content.strip()

    # Return best attempt
    return {
        "results": results,
        "iterations": max_iterations,
        "final_query": query
    }

# Strategy 8: Knowledge Graphs

In [5]:
from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType

# Initialize Graphiti (connects to Neo4j)
graphiti = Graphiti("neo4j://localhost:7687", "neo4j", "password")
async def ingest_document(text: str, source: str):
    """Ingest into knowledge graph"""
    # Graphiti automatically extracts entities and relationships
    await graphiti.add_episode(
        name=source,
        episode_body=text,
        source=EpisodeType.text,
        source_description=f"Document: {source}"
    )
async def search_knowledge_graph(query: str) -> str:
    """Hybrid search: semantic + keyword + graph"""
    # Graphiti combines:
    # - Semantic similarity (embeddings)
    # - BM25 keyword search
    # - Graph structure traversal
    # - Temporal context

    results = await graphiti.search(query=query, num_results=5)

    # Format graph results
    formatted = []
    for result in results:
        formatted.append(
            f"Entity: {result.node.name}\n"
            f"Type: {result.node.type}\n"
            f"Relationships: {result.relationships}"
        )

    return "\n---\n".join(formatted)

# Strategy 9: Hierarchical RAG

In [6]:
def ingest_hierarchical(document: str, title: str):
    """Create parent-child structure"""
    # Parents: large sections (2000 chars)
    parent_chunks = [document[i:i+2000] for i in range(0, len(document), 2000)]

    for parent_id, parent in enumerate(parent_chunks):
        # Store parent
        metadata = {"heading": f"{title} - Section {parent_id}"}
        db.execute(
            "INSERT INTO parent_chunks (id, content, metadata) VALUES (%s, %s, %s)",
            (parent_id, parent, json.dumps(metadata))
        )

        # Children: small chunks (500 chars)
        child_chunks = [parent[j:j+500] for j in range(0, len(parent), 500)]
        for child in child_chunks:
            embedding = get_embedding(child)
            db.execute(
                "INSERT INTO child_chunks (content, embedding, parent_id) VALUES (%s, %s, %s)",
                (child, embedding, parent_id)
            )

async def hierarchical_search(query: str) -> str:
    """Search children, return parents"""
    query_emb = get_embedding(query)

    # Search small children for precision
    results = await db.query(
        """SELECT p.content, p.metadata
           FROM child_chunks c
           JOIN parent_chunks p ON c.parent_id = p.id
           ORDER BY c.embedding  %s LIMIT 3""",
        query_emb
    )

    # Return large parents for context
    formatted = []
    for content, metadata in results:
        meta = json.loads(metadata)
        formatted.append(f"[{meta['heading']}]\n{content}")

    return "\n\n".join(formatted)

# Strategy 10: Late Chunking

In [7]:
def late_chunk(text: str, chunk_size=512) -> list:
    """Embed full document BEFORE chunking"""

    # Step 1: Embed entire document (8192 tokens max)
    full_doc_token_embeddings = transformer_embed(text)  # Token-level

    # Step 2: Define chunk boundaries
    tokens = tokenize(text)
    chunk_boundaries = range(0, len(tokens), chunk_size)

    # Step 3: Pool token embeddings for each chunk
    chunks_with_embeddings = []
    for start in chunk_boundaries:
        end = start + chunk_size
        chunk_text = detokenize(tokens[start:end])

        # Mean pool token embeddings (preserves full doc context!)
        chunk_embedding = mean_pool(full_doc_token_embeddings[start:end])
        chunks_with_embeddings.append((chunk_text, chunk_embedding))

    return chunks_with_embeddings

# Strategy 11: Fine-tuned Embeddings


In [11]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

def prepare_training_data():
    """Domain-specific query-document pairs"""
    return [
        ("What is EBITDA?", "EBITDA (Earnings Before Interest, Taxes..."),
        ("Explain capital expenditure", "Capital expenditure (CapEx) refers to..."),
        # ... thousands more pairs
    ]
def fine_tune_model():
    """Fine-tune on domain data"""
    # Load base model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Prepare training data
    train_examples = prepare_training_data()
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

    # Define loss function
    train_loss = losses.MultipleNegativesRankingLoss(model)

    # Train
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=3,
        warmup_steps=100
    )

    model.save('./fine_tuned_financial_model')
    return model
# Use fine-tuned model
embedding_model = SentenceTransformer('./fine_tuned_financial_model')